# 09_01 EES Uncertainty — Cross-Domain Consolidation

**Reviewer 2 — Comment 4**

This notebook consolidates the completed repeated full-pipeline EES uncertainty analyses for:

- Credit Risk (German Credit)
- Healthcare (Pima Diabetes)
- Criminal Justice (COMPAS)

It uses only the **five common predefined split seeds: 42, 43, 44, 45, 46** for the formal cross-domain comparison.

**Important:** This notebook does **not** retrain models or rerun SHAP, LIME, Stability, or DiCE. It reads the saved repetition-level outputs and creates reviewer-facing cross-domain summaries.


In [1]:
# 0. Setup
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/Master_Research')

COMMON_SEEDS = [42, 43, 44, 45, 46]
EXPECTED_MODELS = 5
EXPECTED_ROWS_PER_DOMAIN = len(COMMON_SEEDS) * EXPECTED_MODELS
EXPECTED_TOTAL_ROWS = EXPECTED_ROWS_PER_DOMAIN * 3

OUTPUT_PATH = (
    ROOT_PATH /
    '07_Results' /
    'cross_domain' /
    '09_01_ees_uncertainty_cross_domain_analysis'
)

TABLES_PATH = OUTPUT_PATH / 'tables'
DATASETS_PATH = OUTPUT_PATH / 'datasets'
LOGS_PATH = OUTPUT_PATH / 'logs'

for p in [OUTPUT_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("Common seeds:", COMMON_SEEDS)
print("Expected rows per domain:", EXPECTED_ROWS_PER_DOMAIN)
print("Expected total rows:", EXPECTED_TOTAL_ROWS)
print("Output:", OUTPUT_PATH)


Mounted at /content/drive
Common seeds: [42, 43, 44, 45, 46]
Expected rows per domain: 25
Expected total rows: 75
Output: /content/drive/MyDrive/Master_Research/07_Results/cross_domain/09_01_ees_uncertainty_cross_domain_analysis


## 1. Locate the three repetition-level result files

The paths below follow the folder structure used by the domain notebooks. If your saved folders differ slightly, edit only the relevant path.


In [2]:
# 1. Input paths
INPUT_FILES = {
    'Credit Risk': (
        ROOT_PATH /
        '07_Results' /
        'german_credit' /
        '09_00_ees_uncertainty_rank_stability' /
        'datasets' /
        'ees_uncertainty_repetition_results.csv'
    ),

    'Healthcare': (
        ROOT_PATH /
        '07_Results' /
        'pima_diabetes' /
        '09_00_ees_uncertainty_rank_stability' /
        'datasets' /
        'ees_uncertainty_repetition_results.csv'
    ),

    'Criminal Justice': (
        ROOT_PATH /
        '07_Results' /
        'compas' /
        '09_00_ees_uncertainty_rank_stability' /
        'datasets' /
        'ees_uncertainty_repetition_results.csv'
    )
}

for domain, path in INPUT_FILES.items():
    print(f"{domain}: {path}")
    print("  Exists:", path.exists())


Credit Risk: /content/drive/MyDrive/Master_Research/07_Results/german_credit/09_00_ees_uncertainty_rank_stability/datasets/ees_uncertainty_repetition_results.csv
  Exists: True
Healthcare: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/09_00_ees_uncertainty_rank_stability/datasets/ees_uncertainty_repetition_results.csv
  Exists: True
Criminal Justice: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/datasets/ees_uncertainty_repetition_results.csv
  Exists: True


## 2. Load, standardise, and retain only seeds 42–46

This step deliberately uses the **first five common predefined seeds**, not the five most favourable results. It also harmonises `regulatory_alignment_score` to `governance_alignment_score` where necessary.


In [3]:
# 2. Load and standardise
domain_frames = []

for domain, path in INPUT_FILES.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find the repetition-level results for {domain}:\n{path}\n"
            "Edit INPUT_FILES above if your output folder differs."
        )

    df = pd.read_csv(path)

    print("\n" + "=" * 80)
    print(domain)
    print("Original rows:", len(df))

    # Accommodate common capitalization variants if needed.
    rename_candidates = {
        'Split_Seed': 'split_seed',
        'Seed': 'split_seed',
        'Repetition': 'repetition',
        'Model_ID': 'model_id',
        'Model_Name': 'model_name',
        'Model': 'model_name',
        'Fidelity_Score': 'fidelity_score',
        'Stability_Score': 'stability_score',
        'Actionability_Score': 'actionability_score',
        'EES_Score': 'ees_score',
        'EES_Rank': 'ees_rank'
    }
    for old, new in rename_candidates.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if (
        'regulatory_alignment_score' in df.columns
        and 'governance_alignment_score' not in df.columns
    ):
        df = df.rename(
            columns={'regulatory_alignment_score': 'governance_alignment_score'}
        )

    required = [
        'split_seed', 'model_name',
        'fidelity_score', 'stability_score',
        'actionability_score', 'ees_score', 'ees_rank'
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"{domain}: missing required columns {missing}.\n"
            f"Available columns:\n{list(df.columns)}"
        )

    # If model_id is absent, create a stable ID from model name.
    if 'model_id' not in df.columns:
        df['model_id'] = df['model_name'].astype(str)

    # Governance is expected to be 1.0 in the present implementation.
    if 'governance_alignment_score' not in df.columns:
        df['governance_alignment_score'] = 1.0
        print("Governance column absent: inserted 1.0 according to current study implementation.")

    print("Available seeds:", sorted(df['split_seed'].dropna().unique().tolist()))

    # Retain only the agreed common seeds.
    df = df[df['split_seed'].isin(COMMON_SEEDS)].copy()

    seed_to_rep = {seed: i + 1 for i, seed in enumerate(COMMON_SEEDS)}
    df['repetition'] = df['split_seed'].map(seed_to_rep)
    df['analysis_domain'] = domain

    print("Rows retained:", len(df))
    print("Seeds retained:", sorted(df['split_seed'].unique().tolist()))
    print("Models retained:", sorted(df['model_name'].astype(str).unique().tolist()))

    # Validation
    if len(df) != EXPECTED_ROWS_PER_DOMAIN:
        raise ValueError(
            f"{domain}: expected {EXPECTED_ROWS_PER_DOMAIN} rows "
            f"(5 seeds × 5 models) but found {len(df)}."
        )

    if df['split_seed'].nunique() != 5:
        raise ValueError(f"{domain}: expected 5 unique seeds.")

    if df['model_name'].nunique() != 5:
        raise ValueError(f"{domain}: expected 5 unique models.")

    # Ensure one model result per seed.
    duplicate_check = df.duplicated(
        subset=['split_seed', 'model_name'], keep=False
    )
    if duplicate_check.any():
        raise ValueError(
            f"{domain}: duplicate seed/model records detected. "
            "Inspect the repetition-level output before continuing."
        )

    domain_frames.append(df)

combined_df = pd.concat(domain_frames, ignore_index=True)

print("\nCombined rows:", len(combined_df))
if len(combined_df) != EXPECTED_TOTAL_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL_ROWS} total rows but found {len(combined_df)}."
    )

display(
    combined_df[
        [
            'analysis_domain', 'repetition', 'split_seed', 'model_name',
            'fidelity_score', 'stability_score', 'actionability_score',
            'governance_alignment_score', 'ees_score', 'ees_rank'
        ]
    ].head(15)
)



Credit Risk
Original rows: 100
Available seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
Rows retained: 25
Seeds retained: [42, 43, 44, 45, 46]
Models retained: ['Decision Tree', 'Logistic Regression', 'MLP', 'Random Forest', 'XGBoost']

Healthcare
Original rows: 100
Available seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
Rows retained: 25
Seeds retained: [42, 43, 44, 45, 46]
Models retained: ['Decision Tree', 'Logistic Regression', 'MLP', 'Random Forest', 'XGBoost']

Criminal Justice
Original rows: 25
Available seeds: [42, 43, 44, 45, 46]
Rows retained: 25
Seeds retained: [42, 43, 44, 45, 46]
Models retained: ['Decision Tree', 'Logistic Regression', 'MLP', 'Random Forest', 'XGBoost']

Combined rows: 75


,analysis_domain,repetition,split_seed,model_name,fidelity_score,stability_score,actionability_score,governance_alignment_score,ees_score,ees_rank
0,Credit Risk,1,42,Logistic Regression,0.826339,1.000000,0.936173,1.0,0.940628,1
1,Credit Risk,1,42,Decision Tree,0.701840,0.918039,0.948544,1.0,0.892106,3
2,Credit Risk,1,42,Random Forest,0.756394,0.978942,0.944807,1.0,0.920036,2
3,Credit Risk,1,42,XGBoost,0.696672,0.000000,0.948515,1.0,0.661297,5
4,Credit Risk,1,42,MLP,0.567939,0.673061,0.946172,1.0,0.796793,4
5,Credit Risk,2,43,Logistic Regression,0.805176,1.000000,0.929515,1.0,0.933673,1
6,Credit Risk,2,43,Decision Tree,0.659834,0.822013,0.933781,1.0,0.853907,3
7,Credit Risk,2,43,Random Forest,0.759204,0.969119,0.942380,1.0,0.917676,2
8,Credit Risk,2,43,XGBoost,0.671273,0.000000,0.942408,1.0,0.653420,5
9,Credit Risk,2,43,MLP,0.602276,0.717234,0.935524,1.0,0.813759,4


## 3. Cross-domain EES uncertainty summary

For five repetitions, the primary uncertainty summaries are:

- mean EES ± sample SD,
- observed minimum and maximum EES,
- mean/median rank,
- observed rank range,
- rank-1 count and frequency.

We deliberately do **not** use 2.5th–97.5th percentile intervals as the main uncertainty summary with only five repetitions.


In [4]:
# 3. Summary
summary_rows = []

for (domain, model_id), g in combined_df.groupby(
    ['analysis_domain', 'model_id'], sort=False
):
    summary_rows.append({
        'Domain': domain,
        'Model_ID': model_id,
        'Model': g['model_name'].iloc[0],
        'N_Repetitions': len(g),

        'Mean_Fidelity': g['fidelity_score'].mean(),
        'SD_Fidelity': g['fidelity_score'].std(ddof=1),

        'Mean_Stability': g['stability_score'].mean(),
        'SD_Stability': g['stability_score'].std(ddof=1),

        'Mean_Actionability': g['actionability_score'].mean(),
        'SD_Actionability': g['actionability_score'].std(ddof=1),

        'Mean_Governance': g['governance_alignment_score'].mean(),

        'Mean_EES': g['ees_score'].mean(),
        'SD_EES': g['ees_score'].std(ddof=1),
        'Min_EES': g['ees_score'].min(),
        'Max_EES': g['ees_score'].max(),

        'Mean_Rank': g['ees_rank'].mean(),
        'Median_Rank': g['ees_rank'].median(),
        'Best_Rank': g['ees_rank'].min(),
        'Worst_Rank': g['ees_rank'].max(),

        'Rank1_Count': int((g['ees_rank'] == 1).sum()),
        'Rank1_Frequency': (g['ees_rank'] == 1).mean()
    })

cross_domain_summary_df = pd.DataFrame(summary_rows)

cross_domain_summary_df = cross_domain_summary_df.sort_values(
    ['Domain', 'Mean_Rank', 'Mean_EES'],
    ascending=[True, True, False]
).reset_index(drop=True)

display(cross_domain_summary_df.round(4))


,Domain,Model_ID,Model,N_Repetitions,Mean_Fidelity,SD_Fidelity,Mean_Stability,SD_Stability,Mean_Actionability,SD_Actionability,...,Mean_EES,SD_EES,Min_EES,Max_EES,Mean_Rank,Median_Rank,Best_Rank,Worst_Rank,Rank1_Count,Rank1_Frequency
0,Credit Risk,logistic_regression,Logistic Regression,5,0.7997,0.0178,0.9991,0.0020,0.9352,0.0057,...,0.9335,0.0052,0.9284,0.9406,1.0,1.0,1,1,5,1.0
1,Credit Risk,random_forest,Random Forest,5,0.7483,0.0141,0.9631,0.0108,0.9418,0.0037,...,0.9133,0.0058,0.9072,0.9200,2.0,2.0,2,2,0,0.0
2,Credit Risk,decision_tree,Decision Tree,5,0.6829,0.0154,0.9421,0.0746,0.9430,0.0081,...,0.8920,0.0221,0.8539,0.9077,3.0,3.0,3,3,0,0.0
3,Credit Risk,mlp,MLP,5,0.5800,0.0145,0.6724,0.0423,0.9421,0.0041,...,0.7986,0.0130,0.7825,0.8138,4.0,4.0,4,4,0,0.0
4,Credit Risk,xgboost,XGBoost,5,0.6774,0.0122,0.0000,0.0000,0.9461,0.0042,...,0.6559,0.0036,0.6519,0.6613,5.0,5.0,5,5,0,0.0
5,Criminal Justice,decision_tree,Decision Tree,5,0.6431,0.0149,1.0000,0.0000,0.9464,0.0040,...,0.8974,0.0045,0.8934,0.9045,1.0,1.0,1,1,5,1.0
6,Criminal Justice,mlp,MLP,5,0.6849,0.0450,0.5833,0.1841,0.9475,0.0031,...,0.8039,0.0444,0.7261,0.8304,2.6,2.0,2,4,0,0.0
7,Criminal Justice,xgboost,XGBoost,5,0.6991,0.0083,0.6355,0.0209,0.9479,0.0034,...,0.8206,0.0061,0.8115,0.8257,2.8,3.0,2,4,0,0.0
8,Criminal Justice,logistic_regression,Logistic Regression,5,0.5886,0.0220,0.7053,0.0228,0.9483,0.0040,...,0.8106,0.0079,0.7987,0.8191,3.6,4.0,3,4,0,0.0
9,Criminal Justice,random_forest,Random Forest,5,0.7026,0.0126,0.0000,0.0000,0.9472,0.0029,...,0.6624,0.0029,0.6593,0.6657,5.0,5.0,5,5,0,0.0


## 4. Champion frequency across the five common splits

In [5]:
# 4. Champion frequency
champion_frequency_df = (
    combined_df
    .assign(is_champion=lambda x: x['ees_rank'] == 1)
    .groupby(
        ['analysis_domain', 'model_id', 'model_name'],
        as_index=False
    )
    .agg(
        Rank1_Count=('is_champion', 'sum'),
        N_Repetitions=('repetition', 'nunique')
    )
)

champion_frequency_df['Rank1_Frequency'] = (
    champion_frequency_df['Rank1_Count'] /
    champion_frequency_df['N_Repetitions']
)

champion_frequency_df = champion_frequency_df.sort_values(
    ['analysis_domain', 'Rank1_Count', 'model_name'],
    ascending=[True, False, True]
).reset_index(drop=True)

display(champion_frequency_df)


,analysis_domain,model_id,model_name,Rank1_Count,N_Repetitions,Rank1_Frequency
0,Credit Risk,logistic_regression,Logistic Regression,5,5,1.0
1,Credit Risk,decision_tree,Decision Tree,0,5,0.0
2,Credit Risk,mlp,MLP,0,5,0.0
3,Credit Risk,random_forest,Random Forest,0,5,0.0
4,Credit Risk,xgboost,XGBoost,0,5,0.0
5,Criminal Justice,decision_tree,Decision Tree,5,5,1.0
6,Criminal Justice,logistic_regression,Logistic Regression,0,5,0.0
7,Criminal Justice,mlp,MLP,0,5,0.0
8,Criminal Justice,random_forest,Random Forest,0,5,0.0
9,Criminal Justice,xgboost,XGBoost,0,5,0.0


## 5. Full rank matrix by seed

In [6]:
# 5. Rank matrix
rank_matrix_df = combined_df.pivot_table(
    index=['analysis_domain', 'model_name'],
    columns='split_seed',
    values='ees_rank',
    aggfunc='first'
).reset_index()

rank_matrix_df.columns.name = None

rank_matrix_df = rank_matrix_df.rename(
    columns={
        42: 'Seed_42',
        43: 'Seed_43',
        44: 'Seed_44',
        45: 'Seed_45',
        46: 'Seed_46'
    }
)

display(rank_matrix_df)


,analysis_domain,model_name,Seed_42,Seed_43,Seed_44,Seed_45,Seed_46
0,Credit Risk,Decision Tree,3,3,3,3,3
1,Credit Risk,Logistic Regression,1,1,1,1,1
2,Credit Risk,MLP,4,4,4,4,4
3,Credit Risk,Random Forest,2,2,2,2,2
4,Credit Risk,XGBoost,5,5,5,5,5
5,Criminal Justice,Decision Tree,1,1,1,1,1
6,Criminal Justice,Logistic Regression,3,3,4,4,4
7,Criminal Justice,MLP,2,4,3,2,2
8,Criminal Justice,Random Forest,5,5,5,5,5
9,Criminal Justice,XGBoost,4,2,2,3,3


## 6. EES champion in each domain and split

In [7]:
# 6. Split champions
split_champions_df = (
    combined_df[
        combined_df['ees_rank'] == 1
    ][
        [
            'analysis_domain',
            'repetition',
            'split_seed',
            'model_name',
            'ees_score'
        ]
    ]
    .sort_values(['analysis_domain', 'split_seed'])
    .reset_index(drop=True)
)

display(split_champions_df)


,analysis_domain,repetition,split_seed,model_name,ees_score
0,Credit Risk,1,42,Logistic Regression,0.940628
1,Credit Risk,2,43,Logistic Regression,0.933673
2,Credit Risk,3,44,Logistic Regression,0.928354
3,Credit Risk,4,45,Logistic Regression,0.928785
4,Credit Risk,5,46,Logistic Regression,0.936126
5,Criminal Justice,1,42,Decision Tree,0.904530
6,Criminal Justice,2,43,Decision Tree,0.896463
7,Criminal Justice,3,44,Decision Tree,0.893403
8,Criminal Justice,4,45,Decision Tree,0.893753
9,Criminal Justice,5,46,Decision Tree,0.898658


## 7. Dimension-level variability

This table helps identify whether repeated-split variation is being driven mainly by Fidelity, Stability, Counterfactual Actionability, or the final EES.


In [8]:
# 7. Dimension-level variability
dimension_summary_df = (
    combined_df
    .groupby(
        ['analysis_domain', 'model_name'],
        as_index=False
    )
    .agg(
        Fidelity_Mean=('fidelity_score', 'mean'),
        Fidelity_SD=('fidelity_score', 'std'),

        Stability_Mean=('stability_score', 'mean'),
        Stability_SD=('stability_score', 'std'),

        Actionability_Mean=('actionability_score', 'mean'),
        Actionability_SD=('actionability_score', 'std'),

        Governance_Mean=('governance_alignment_score', 'mean'),

        EES_Mean=('ees_score', 'mean'),
        EES_SD=('ees_score', 'std')
    )
)

display(dimension_summary_df.round(4))


,analysis_domain,model_name,Fidelity_Mean,Fidelity_SD,Stability_Mean,Stability_SD,Actionability_Mean,Actionability_SD,Governance_Mean,EES_Mean,EES_SD
0,Credit Risk,Decision Tree,0.6829,0.0154,0.9421,0.0746,0.9430,0.0081,1.0,0.8920,0.0221
1,Credit Risk,Logistic Regression,0.7997,0.0178,0.9991,0.0020,0.9352,0.0057,1.0,0.9335,0.0052
2,Credit Risk,MLP,0.5800,0.0145,0.6724,0.0423,0.9421,0.0041,1.0,0.7986,0.0130
3,Credit Risk,Random Forest,0.7483,0.0141,0.9631,0.0108,0.9418,0.0037,1.0,0.9133,0.0058
4,Credit Risk,XGBoost,0.6774,0.0122,0.0000,0.0000,0.9461,0.0042,1.0,0.6559,0.0036
5,Criminal Justice,Decision Tree,0.6431,0.0149,1.0000,0.0000,0.9464,0.0040,1.0,0.8974,0.0045
6,Criminal Justice,Logistic Regression,0.5886,0.0220,0.7053,0.0228,0.9483,0.0040,1.0,0.8106,0.0079
7,Criminal Justice,MLP,0.6849,0.0450,0.5833,0.1841,0.9475,0.0031,1.0,0.8039,0.0444
8,Criminal Justice,Random Forest,0.7026,0.0126,0.0000,0.0000,0.9472,0.0029,1.0,0.6624,0.0029
9,Criminal Justice,XGBoost,0.6991,0.0083,0.6355,0.0209,0.9479,0.0034,1.0,0.8206,0.0061


## 8. Compact reviewer/manuscript table

In [9]:
# 8. Compact manuscript-ready summary
manuscript_table_df = cross_domain_summary_df[
    [
        'Domain',
        'Model',
        'Mean_EES',
        'SD_EES',
        'Best_Rank',
        'Worst_Rank',
        'Rank1_Count',
        'Rank1_Frequency'
    ]
].copy()

manuscript_table_df['EES_Mean_SD'] = (
    manuscript_table_df['Mean_EES'].map(lambda x: f"{x:.3f}")
    + " ± "
    + manuscript_table_df['SD_EES'].map(lambda x: f"{x:.3f}")
)

manuscript_table_df['Rank_Range'] = (
    manuscript_table_df['Best_Rank'].astype(int).astype(str)
    + "–"
    + manuscript_table_df['Worst_Rank'].astype(int).astype(str)
)

manuscript_table_df['Rank1'] = (
    manuscript_table_df['Rank1_Count'].astype(int).astype(str)
    + "/5 ("
    + (manuscript_table_df['Rank1_Frequency'] * 100)
        .round(0).astype(int).astype(str)
    + "%)"
)

manuscript_table_df = manuscript_table_df[
    ['Domain', 'Model', 'EES_Mean_SD', 'Rank_Range', 'Rank1']
]

display(manuscript_table_df)


,Domain,Model,EES_Mean_SD,Rank_Range,Rank1
0,Credit Risk,Logistic Regression,0.934 ± 0.005,1–1,5/5 (100%)
1,Credit Risk,Random Forest,0.913 ± 0.006,2–2,0/5 (0%)
2,Credit Risk,Decision Tree,0.892 ± 0.022,3–3,0/5 (0%)
3,Credit Risk,MLP,0.799 ± 0.013,4–4,0/5 (0%)
4,Credit Risk,XGBoost,0.656 ± 0.004,5–5,0/5 (0%)
5,Criminal Justice,Decision Tree,0.897 ± 0.005,1–1,5/5 (100%)
6,Criminal Justice,MLP,0.804 ± 0.044,2–4,0/5 (0%)
7,Criminal Justice,XGBoost,0.821 ± 0.006,2–4,0/5 (0%)
8,Criminal Justice,Logistic Regression,0.811 ± 0.008,3–4,0/5 (0%)
9,Criminal Justice,Random Forest,0.662 ± 0.003,5–5,0/5 (0%)


## 9. Cross-domain champion stability summary

This produces one row per domain for quick interpretation.


In [10]:
# 9. Domain-level champion stability
domain_champion_summary = []

for domain, g in combined_df.groupby('analysis_domain'):

    champs = g[g['ees_rank'] == 1]

    counts = (
        champs['model_name']
        .value_counts()
        .sort_values(ascending=False)
    )

    modal_champion = counts.index[0]
    wins = int(counts.iloc[0])

    domain_champion_summary.append({
        'Domain': domain,
        'Most_Frequent_Champion': modal_champion,
        'Rank1_Count': wins,
        'Rank1_Frequency': wins / 5,
        'Unique_Champions': champs['model_name'].nunique(),
        'Champion_Stable_5_of_5': wins == 5
    })

domain_champion_summary_df = pd.DataFrame(domain_champion_summary)

display(domain_champion_summary_df)


,Domain,Most_Frequent_Champion,Rank1_Count,Rank1_Frequency,Unique_Champions,Champion_Stable_5_of_5
0,Credit Risk,Logistic Regression,5,1.0,1,True
1,Criminal Justice,Decision Tree,5,1.0,1,True
2,Healthcare,Logistic Regression,4,0.8,2,False


## 10. Save all consolidated outputs

In [11]:
# 10. Save outputs
combined_df.to_csv(
    DATASETS_PATH / 'ees_uncertainty_common_5_seeds_combined.csv',
    index=False
)

cross_domain_summary_df.to_csv(
    TABLES_PATH / 'ees_uncertainty_cross_domain_summary.csv',
    index=False
)

champion_frequency_df.to_csv(
    TABLES_PATH / 'ees_champion_frequency.csv',
    index=False
)

rank_matrix_df.to_csv(
    TABLES_PATH / 'ees_rank_matrix.csv',
    index=False
)

split_champions_df.to_csv(
    TABLES_PATH / 'ees_split_champions.csv',
    index=False
)

dimension_summary_df.to_csv(
    TABLES_PATH / 'ees_dimension_variability.csv',
    index=False
)

manuscript_table_df.to_csv(
    TABLES_PATH / 'ees_uncertainty_manuscript_table.csv',
    index=False
)

domain_champion_summary_df.to_csv(
    TABLES_PATH / 'ees_domain_champion_stability.csv',
    index=False
)

excel_path = TABLES_PATH / 'ees_uncertainty_cross_domain_analysis.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    combined_df.to_excel(
        writer, sheet_name='all_results', index=False
    )

    cross_domain_summary_df.to_excel(
        writer, sheet_name='summary', index=False
    )

    champion_frequency_df.to_excel(
        writer, sheet_name='champion_frequency', index=False
    )

    rank_matrix_df.to_excel(
        writer, sheet_name='rank_matrix', index=False
    )

    split_champions_df.to_excel(
        writer, sheet_name='split_champions', index=False
    )

    dimension_summary_df.to_excel(
        writer, sheet_name='dimension_variability', index=False
    )

    manuscript_table_df.to_excel(
        writer, sheet_name='manuscript_table', index=False
    )

    domain_champion_summary_df.to_excel(
        writer, sheet_name='domain_champion_stability', index=False
    )

manifest = {
    'analysis': 'Reviewer 2 Comment 4 - cross-domain EES uncertainty',
    'run_timestamp': datetime.now().isoformat(),
    'common_split_seeds': COMMON_SEEDS,
    'n_repetitions_per_domain': 5,
    'n_models_per_domain': 5,
    'n_domains': 3,
    'expected_total_model_level_records': EXPECTED_TOTAL_ROWS,
    'selection_rule': (
        'First five common predefined split seeds (42-46); '
        'not selected according to observed results.'
    ),
    'uncertainty_reporting': (
        'Mean and sample SD of EES, observed rank range, and rank-1 '
        'frequency across five repeated stratified full-pipeline splits.'
    ),
    'interpretation_note': (
        'Five repetitions provide a targeted repeated-split robustness '
        'assessment. They are not presented as a precise population-level '
        'confidence interval.'
    )
}

with open(
    LOGS_PATH / 'ees_uncertainty_cross_domain_manifest.json',
    'w'
) as f:
    json.dump(manifest, f, indent=4)

print("Saved combined dataset:", DATASETS_PATH)
print("Saved tables:", TABLES_PATH)
print("Saved workbook:", excel_path)


Saved combined dataset: /content/drive/MyDrive/Master_Research/07_Results/cross_domain/09_01_ees_uncertainty_cross_domain_analysis/datasets
Saved tables: /content/drive/MyDrive/Master_Research/07_Results/cross_domain/09_01_ees_uncertainty_cross_domain_analysis/tables
Saved workbook: /content/drive/MyDrive/Master_Research/07_Results/cross_domain/09_01_ees_uncertainty_cross_domain_analysis/tables/ees_uncertainty_cross_domain_analysis.xlsx


## Interpretation checklist

After running the notebook, bring back these four outputs for manuscript/reviewer interpretation:

1. **`manuscript_table`** — mean ± SD, rank range, rank-1 frequency.
2. **`rank_matrix`** — exact rank for every model under seeds 42–46.
3. **`split_champions`** — champion in each repeated split.
4. **`dimension_variability`** — helps identify which EES dimensions contribute most to repeated-split variability.

The results should be reported exactly as observed. Rank instability, if present, is itself an uncertainty result and should be disclosed rather than suppressed.
